# Understand cohorts, missing outcomes, and regime changes

Upload the complete project ZIP if using Colab. Use Python 3.12 for the verified dependency lock. Run all cells. The original dashboard model is a frozen historical demonstration, not the preferred research method.

In [1]:
import sys, os, json, subprocess, tempfile, zipfile
from pathlib import Path
from IPython.display import display, Markdown
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
ROOT = next((p for p in [Path.cwd(), Path.cwd().parent, Path('/content/delaylens')] if (p/'experiments/REVIEW_PROTOCOL.md').exists()), None)
if ROOT is None and IN_COLAB:
    from google.colab import files
    uploads = files.upload()
    archive = next(name for name in uploads if name.endswith('.zip'))
    with zipfile.ZipFile(archive) as z:
        for name in z.namelist():
            parts = Path(name).parts
            if not parts or parts[0] != 'delaylens' or '..' in parts or Path(name).is_absolute():
                raise ValueError('Unexpected archive member')
        z.extractall('/content')
    ROOT = Path('/content/delaylens')
if ROOT is None: raise RuntimeError('Open this notebook inside the extracted DelayLens project.')
os.chdir(ROOT)
PYTHON = sys.executable
if IN_COLAB:
    env = Path('/content/delaylens-review-env')
    if not (env/'bin/python').exists():
        subprocess.run([sys.executable,'-m','venv',str(env)],check=True)
    PYTHON = str(env/'bin/python')
    subprocess.run([PYTHON,'-m','pip','install','-r','requirements-lock.txt'],cwd=ROOT,check=True)
def run_python(source):
    result = subprocess.run([PYTHON,'-c',source],cwd=ROOT,text=True,capture_output=True)
    print(result.stdout)
    if result.returncode:
        print(result.stderr)
        result.check_returncode()
print('Ready. This notebook reads real saved evidence; it does not refit candidates.')


Ready. This notebook reads real saved evidence; it does not refit candidates.


## Approval-time cohort
Rows enter the queue using input eligibility, never eventual status or delivery. A missing or impossible delivery is unknown. It still consumes a review slot if selected. Known-hit lift is not full-population effectiveness.

In [2]:
print((ROOT/'reports/review/cohort_audit.json').read_text())
print((ROOT/'reports/review/raw_audit.json').read_text())

{
  "raw_orders": 99441,
  "status_counts": {
    "delivered": 96478,
    "shipped": 1107,
    "canceled": 625,
    "unavailable": 609,
    "invoiced": 314,
    "processing": 301,
    "created": 5,
    "approved": 2
  },
  "source_sha256": {
    "olist_orders_dataset.csv": "8df58ef3d2d7e9944010f7beecd9b75367f5588ec6e3c91cec19ae3345ef9ecf",
    "olist_order_items_dataset.csv": "0bc4d068c4fe38cbb01bd90e8746e3c613fe7b4baef75fab7b0e329701c3e279",
    "olist_customers_dataset.csv": "983a422239e1712ded753b3bf9ecf47dc73f144d306029dcfa99e70a226883d2",
    "olist_sellers_dataset.csv": "1f643d2b950373b85735e7794b20986f528d7a000432e7c6f9bcbb44d0846a0e",
    "olist_products_dataset.csv": "3e6569628a17fbc75fd206ee357b59e20364b9afa90f5b6cd5b4d624c58aa9cc",
    "product_category_name_translation.csv": "a81f0d1f27b27e7293f761bc79e3ce8f348ee39c4b3ed3e49bde38f478586278"
  },
  "exclusions": {
    "missing_required_timestamp": 160,
    "not_exactly_one_seller": 1907,
    "input_contract_failure": 16
  },

## Raw monthly evidence
These rows include all raw orders with approval in each month. The prediction cohort is smaller because of input eligibility. Status is descriptive, not an input.

In [3]:
run_python("import pandas as pd\nx=pd.read_csv('reports/review/raw_monthly.csv')\nprint(x[['month','orders','median_promised_days','unknown_outcomes','delivered','shipped','canceled','unavailable']].to_string(index=False))")

  month  orders  median_promised_days  unknown_outcomes  delivered  shipped  canceled  unavailable
2017-01     760                  39.0                45        715       16         2            8
2017-02    1765                  31.0               127       1638       21        15           47
2017-03    2689                  23.0               136       2554       45        31           32
2017-04    2374                  26.0                96       2278       49        13            9
2017-05    3693                  24.0               146       3548       54        19           32
2017-06    3252                  24.0               109       3143       48        14           24
2017-07    3974                  23.0               147       3828       54        23           52
2017-08    4348                  23.0               131       4217       41        21           31
2017-09    4301                  22.0               139       4170       40        16           37
2017-10   

## Missing-outcome bounds
Lower precision = confirmed late selected / all selected. Upper precision also counts selected unknowns as late. Neither bound is an estimate of savings. The missing outcomes may be non-random.

In [4]:
run_python("import pandas as pd\nx=pd.read_csv('reports/review/stress_summary.csv')\nprint(x[['method','budget','selected','confirmed_late_found','selected_unknown','precision_lower','precision_upper']].to_string(index=False))")

             method  budget  selected  confirmed_late_found  selected_unknown  precision_lower  precision_upper
 short_promise_rule      10       569                    80                32         0.140598         0.196837
 short_promise_rule      25      1397                   141                49         0.100931         0.136006
 short_promise_rule      50      2772                   250                87         0.090188         0.121573
route_duration_rule      10       569                    60                32         0.105448         0.161687
route_duration_rule      25      1397                   139                49         0.099499         0.134574
route_duration_rule      50      2772                   241                88         0.086941         0.118687
 frozen_original_ml      10       569                    53                21         0.093146         0.130053
 frozen_original_ml      25      1397                    93                45         0.066571         0